
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Windowed Aggregation with Watermark

## Objectives
1. Build streaming DataFrames with time window aggregates with watermark
1. Write streaming query results to Delta table using `update` mode and `forEachBatch()`
1. Monitor the streaming query


### Classes
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.html" target="_blank">DataStreamReader</a>
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.html" target="_blank">DataStreamWriter</a>
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.html" target="_blank">StreamingQuery</a>
- <a href="https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html#using-foreach-and-foreachbatch" target="_blank">foreachbatch</a>
- <a href="https://docs.databricks.com/en/structured-streaming/delta-lake.html#upsert-from-streaming-queries-using-foreachbatch&language-python" target="_blank">update mode with foreachbatch</a>

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

In [0]:
%run ./Includes/Classroom-Setup-04

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



## Build Streaming DataFrames

Obtain an initial streaming DataFrame from a Delta-format file source.

In [0]:
from pyspark.sql.functions import window, sum, col
from pyspark.sql.types import TimestampType

parsed_df = (spark.readStream
                    .load('/Volumes/dbacademy_ecommerce/v01/delta/events_hist')
                    .withColumn("event_timestamp", (col("event_timestamp") / 1e6).cast("timestamp"))
                    .withColumn("event_previous_timestamp", (col("event_previous_timestamp") / 1e6).cast("timestamp"))

                    # filter out zero revenue events
                    .filter("ecommerce.purchase_revenue_in_usd IS NOT NULL AND ecommerce.purchase_revenue_in_usd != 0")
)

In [0]:
display(parsed_df)

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Windows,"List(1795.0, 1, 1)",finalize,2020-07-01T17:02:24.33906Z,2020-07-01T17:07:28.104571Z,"List(Corpus Christi, TX)","List(List(null, M_PREM_Q, Premium Queen Mattress, 1795.0, 1795.0, 1))",facebook,1593616330759799,UA000000106537762
macOS,"List(1045.0, 1, 1)",finalize,2020-07-01T14:47:40.452264Z,2020-07-01T14:50:04.57077Z,"List(Kearney, NE)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593610742063884,UA000000106509088
Windows,"List(535.5, 1, 1)",finalize,2020-07-03T22:46:10.279527Z,2020-07-03T22:47:13.585044Z,"List(Phoenix, AZ)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1593608758511600,UA000000106500611
Android,"List(1095.0, 1, 1)",finalize,2020-07-01T16:26:16.166531Z,2020-07-01T16:40:12.951939Z,"List(Albuquerque, NM)","List(List(null, M_PREM_T, Premium Twin Mattress, 1095.0, 1095.0, 1))",google,1593616245128899,UA000000106537277
Windows,"List(940.5, 1, 1)",finalize,2020-07-03T20:05:00.141558Z,2020-07-03T20:17:10.173589Z,"List(Dallas, TX)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593606856891824,UA000000106493435
macOS,"List(1995.0, 1, 1)",finalize,2020-07-01T07:22:08.101005Z,2020-07-01T07:22:49.734017Z,"List(Clearwater, FL)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",email,1593585605089308,UA000000106460128
macOS,"List(940.5, 1, 1)",finalize,2020-07-02T10:25:22.565145Z,2020-07-02T10:35:10.618872Z,"List(Kansas City, MO)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593603980046403,UA000000106484209
macOS,"List(1045.0, 1, 1)",finalize,2020-07-01T11:27:05.823702Z,2020-07-01T12:38:58.856941Z,"List(Breckenridge, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593598423581032,UA000000106471468
iOS,"List(1695.0, 1, 1)",finalize,2020-07-01T15:09:30.661327Z,2020-07-01T15:09:35.423652Z,"List(Newark, NJ)","List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))",instagram,1593613743747321,UA000000106523740
iOS,"List(1045.0, 1, 1)",finalize,2020-07-01T14:08:01.92971Z,2020-07-01T14:16:13.155915Z,"List(Raleigh, NC)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593610433648747,UA000000106507739


In [0]:
# now add up revenues by city by 60 minute time window
windowed_df = (parsed_df
                     # The watermark sets up the timing window. The watermark says "I'll accept events up to 90 minutes late." So if a window closes at 3:00 PM, Spark will still accept events timestamped as late as 4:30 PM before finalizing results.
                    .withWatermark(eventTime="event_timestamp", delayThreshold="90 minutes")
                    # group by city by hour
                    # For each 60-minute window and city combination, Spark groups matching records
                    .groupBy(window(timeColumn="event_timestamp", windowDuration="60 minutes"), "geo.city")
                    .agg(sum("ecommerce.purchase_revenue_in_usd").alias("total_revenue"))

)

In [0]:
display(windowed_df)

window,city,total_revenue
"List(2020-07-02T20:00:00Z, 2020-07-02T21:00:00Z)",Azusa,1615.5
"List(2020-06-21T05:00:00Z, 2020-06-21T06:00:00Z)",Foley,535.5
"List(2020-06-26T21:00:00Z, 2020-06-26T22:00:00Z)",Mobile,595.0
"List(2020-06-29T21:00:00Z, 2020-06-29T22:00:00Z)",Frisco,1195.0
"List(2020-06-27T04:00:00Z, 2020-06-27T05:00:00Z)",Fresno,903.6
"List(2020-06-20T16:00:00Z, 2020-06-20T17:00:00Z)",Joliet,595.0
"List(2020-07-03T20:00:00Z, 2020-07-03T21:00:00Z)",Seattle,59.0
"List(2020-06-24T16:00:00Z, 2020-06-24T17:00:00Z)",Chicago,1795.0
"List(2020-06-20T20:00:00Z, 2020-06-20T21:00:00Z)",Shawnee,1615.5
"List(2020-06-29T21:00:00Z, 2020-06-29T22:00:00Z)",Madison,1795.0


## Write streaming results

Let's explore a couple options for writing the results.

### Write streaming results in `append` mode (option 1)

In the below example, the sink table is appended with new rows from results table on each trigger.
Think about how append mode writing data to sink. What will happen to records of those city whose hourly revenue got updated due to late arrival data. Would it be updated into the sink with "append" mode?

In [0]:
checkpoint_path = f"{DA.paths.working_dir}/query_revenue_by_city_by_hour_append"
# Write the output of a streaming aggregation query into Delta table as updates.The implication of append output modes in the context of window aggregation and watermarks is that an aggregate can be produced only once and can not be updated. Therefore, once the aggregate is produced, the engine can delete the aggregate's state and thus keep the overall aggregation state bounded.
windowed_query = (windowed_df.writeStream
                  .queryName("query_revenue_by_city_by_hour_append")
                  .option("checkpointLocation", checkpoint_path)
                  .trigger(availableNow=True)
                  .outputMode("append")
                  .table("revenue_by_city_by_hour_append")
                )

In [0]:
%sql
SELECT * FROM revenue_by_city_by_hour_append

window,city,total_revenue
"List(2020-07-02T20:00:00Z, 2020-07-02T21:00:00Z)",Azusa,1615.5
"List(2020-06-21T05:00:00Z, 2020-06-21T06:00:00Z)",Foley,535.5
"List(2020-06-26T21:00:00Z, 2020-06-26T22:00:00Z)",Mobile,595.0
"List(2020-06-29T21:00:00Z, 2020-06-29T22:00:00Z)",Frisco,1195.0
"List(2020-06-27T04:00:00Z, 2020-06-27T05:00:00Z)",Fresno,903.6
"List(2020-06-20T16:00:00Z, 2020-06-20T17:00:00Z)",Joliet,595.0
"List(2020-07-03T20:00:00Z, 2020-07-03T21:00:00Z)",Seattle,59.0
"List(2020-06-24T16:00:00Z, 2020-06-24T17:00:00Z)",Chicago,1795.0
"List(2020-06-20T20:00:00Z, 2020-06-20T21:00:00Z)",Shawnee,1615.5
"List(2020-06-29T21:00:00Z, 2020-06-29T22:00:00Z)",Madison,1795.0


In [0]:
%sql
DESCRIBE HISTORY revenue_by_city_by_hour_append

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-01-15T22:38:10Z,75487044008882,labuser13429858_1768515742@vocareum.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2298896422562451),0115-222246-9gscgu1e,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 128, numRemovedBytes -> 366607, p25FileSize -> 41287, numDeletionVectorsRemoved -> 0, minFileSize -> 41287, numAddedFiles -> 1, maxFileSize -> 41287, p75FileSize -> 41287, p50FileSize -> 41287, numAddedBytes -> 41287)",null,Databricks-Runtime/17.3.x-scala2.13
2,2026-01-15T22:37:36Z,75487044008882,labuser13429858_1768515742@vocareum.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 2bf3819d-31fa-4770-ac22-a4b94b3c9355, epochId -> 1, statsOnLoad -> false)",null,List(2298896422562451),0115-222246-9gscgu1e,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 8470, numOutputBytes -> 572928, numAddedFiles -> 200)",null,Databricks-Runtime/17.3.x-scala2.13
1,2026-01-15T22:37:20Z,75487044008882,labuser13429858_1768515742@vocareum.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 2bf3819d-31fa-4770-ac22-a4b94b3c9355, epochId -> 0, statsOnLoad -> false)",null,List(2298896422562451),0115-222246-9gscgu1e,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0, numAddedFiles -> 0)",null,Databricks-Runtime/17.3.x-scala2.13
0,2026-01-15T22:36:34Z,75487044008882,labuser13429858_1768515742@vocareum.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.writePartitionColumnsToParquet"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e1d3dcb2-9ef4-47fc-9ffd-8405ea64694b"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-deaee925-0565-4e0c-af30-63db31ceb80e""}, statsOnLoad -> false)",null,List(2298896422562451),0115-222246-9gscgu1e,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-scala2.13



### Write streaming query results in `update` mode (option 2)

Take the final streaming DataFrame (our result table) and write it to a Delta Table sink in `update` mode. This approach gives much greater control to the developer when it comes to updating the sink, albeit with greater complexity.

**NOTE:** The syntax for Writing streaming results to a Delta table or dataset in `update` mode is a little different. It requires use of the `MERGE` command within a `forEachBatch()` function call. This also requires the target table to be pre-created.


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, ArrayType, TimestampType

# Here we are creating the table with the same schema as the incoming dataframe
schema = StructType([StructField('window',StructType([StructField('start', TimestampType(), True), 
                                                      StructField('end', TimestampType(), True)]), False), 
                     StructField('city', StringType(), True), 
                     StructField('total_revenue', DoubleType(), True)])

empty_df = spark.createDataFrame([], schema=schema)

empty_df.write.saveAsTable(name="revenue_by_city_by_hour", mode='overwrite')

In [0]:
# Function to upsert microBatchOutputDF into Delta table using merge
# upserts (updates or inserts) data into a table 
def upsertToDelta(microBatchOutputDF, batchId):
  # Set the dataframe to view name
  microBatchOutputDF.createOrReplaceTempView("updates")
  # IMP: You have to use the SparkSession that has been used to define the `updates` dataframe

  # In Databricks Runtime 10.5 and below, you must use the following:
  # microBatchOutputDF._jdf.sparkSession().sql("""
  microBatchOutputDF.sparkSession.sql("""
    MERGE INTO revenue_by_city_by_hour t
    USING updates s
    ON t.window.start = s.window.start AND t.window.end = s.window.end AND t.city = s.city
    WHEN MATCHED THEN UPDATE SET * --If a matching row exists (meaning you've already seen this city/time window combo before), replace all its values with the new incoming data. The * means "all columns."
    WHEN NOT MATCHED THEN INSERT * --If no matching row exists (this is the first time seeing this city/time window combo), insert the entire row as a new record.
  """)

Check the `revenue_by_city_by_hour` table before writing it to a Delta table sink in `UPDATE` mode.

In [0]:
%sql
SELECT * FROM revenue_by_city_by_hour

window,city,total_revenue


### Execute below code to write result table into delta table using **update** mode
Dive deep into the query raw metrics and pay close attention to stateful operator section, see if you could identify **watermark** work in action and number of rows removed due to it's settings

In [0]:
checkpoint_path = f"{DA.paths.working_dir}/query_revenue_by_city_by_hour"

# Write the output of a streaming aggregation query into Delta table as updates
windowed_query = (windowed_df.writeStream
                  .foreachBatch(upsertToDelta)
                  .outputMode("update")
                  .queryName("query_revenue_by_city_by_hour")
                  .option("checkpointLocation", checkpoint_path)
                  .trigger(availableNow=True)
                  .start()
                )

In [0]:
%sql
SELECT * FROM revenue_by_city_by_hour

window,city,total_revenue


In [0]:
%sql
DESCRIBE HISTORY revenue_by_city_by_hour

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-01-15T23:04:35Z,75487044008882,labuser13429858_1768515742@vocareum.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2298896422562451),0115-222246-9gscgu1e,0,WriteSerializable,false,"Map(numFiles -> 0, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 0, numOutputBytes -> 0)",null,Databricks-Runtime/17.3.x-scala2.13
0,2026-01-15T22:37:22Z,75487044008882,labuser13429858_1768515742@vocareum.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2298896422562451),0115-222246-9gscgu1e,null,WriteSerializable,false,"Map(numFiles -> 0, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 0, numOutputBytes -> 0)",null,Databricks-Runtime/17.3.x-scala2.13


In [0]:
for s in spark.streams.active:
  print(s.name)
  s.stop()

display_query_11
display_query_10
query_revenue_by_city_by_hour


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>